# Imports

In [ ]:
import numpy as np
import tensorflow as tf
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, fbeta_score, accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import optuna
import os
import random
from mamba_ssm import Mamba2
from sklearn.preprocessing import StandardScaler, RobustScaler
import pandas as pd
import json

# GPU Resources control

In [ ]:
if torch.cuda.is_available():    
    print('There are %d GPU(s) available.' % torch.cuda.device_count())
    print('We will use the GPU:', torch.cuda.get_device_name(0))

else:
    print('No GPU available, using the CPU instead.')
   
print("\n\nCHECKING FOR TENSORFLOW")
print(tf.config.list_physical_devices('GPU'))

gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    print(e)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"--- Usando dispositivo: {DEVICE} ---")

# Setting randomness for reproducibility

In [ ]:
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Defining Experiments

In [ ]:
EXPERIMENTS_CONFIG = [
    {
        "name": "Cap 51",
        "csv_file": "cap51processed.csv",
        "train_start_index": 0,
        "train_end_index": 2934,
        "anomaly_start_index": 5632,
        "optuna_trials": 100,
        "z_scores_to_test": [12.5, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0, 45.0, 50.0, 55.0, 60.0, 70.0]
    },
    {
        "name": "Cap 52",
        "csv_file": "cap52processed.csv",
        "train_start_index": 0,
        "train_end_index": 324,
        "anomaly_start_index": 778,
        "optuna_trials": 200,
        "z_scores_to_test": [70.0, 75.0, 80.0, 85.0, 90.0, 95.0, 100.0, 110.0, 120.0, 125.0, 130.0, 140.0, 150.0, 160.0]
    },
    {
        "name": "Mentored",
        "csv_file": "mentored.csv",
        "train_start_index": 0,
        "train_end_index": 1859,
        "anomaly_start_index": 3601,
        "optuna_trials": 100,
        "z_scores_to_test": [3.0, 5.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 10.0, 15.0]
    }
]

# Model Definition

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm_x = x.norm(2, dim=-1, keepdim=True)
        rms_x = norm_x * (x.shape[-1] ** -0.5)
        x_normed = x / (rms_x + self.eps)
        return x_normed * self.weight

class GatedMambaBlock(nn.Module):
    def __init__(self, d_model, d_state, dropout_rate, headdim=8):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.mamba = Mamba2(d_model=d_model, d_state=d_state, d_conv=4, expand=2, headdim=headdim)
        self.gate = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        residual = x
        x_norm = self.norm(x)
        mamba_out = self.mamba(x_norm)
        gate_val = torch.sigmoid(self.gate(x_norm))
        gated_out = mamba_out * gate_val
        dropped_out = self.dropout(gated_out)
        return dropped_out + residual

class MambaAnomalyDetector(nn.Module):
    def __init__(self, input_dim, d_model, d_state, num_layers, dropout_rate, headdim=8):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.mamba_blocks = nn.ModuleList([
            GatedMambaBlock(d_model=d_model, d_state=d_state, dropout_rate=dropout_rate, headdim=headdim)
            for _ in range(num_layers)
        ])
        self.post_mamba_norm = RMSNorm(d_model)
        self.predictor = nn.Linear(d_model, input_dim * 2)

    def forward(self, x):
        x_proj = self.input_projection(x)
        for block in self.mamba_blocks:
            x_proj = block(x_proj)
        x_final = self.post_mamba_norm(x_proj[:, -1, :])
        prediction = self.predictor(x_final)
        mean, log_var = torch.chunk(prediction, 2, dim=-1)
        return mean, log_var

class UncertaintyLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, y_pred_mean, y_pred_log_var, y_true):
        loss = torch.exp(-y_pred_log_var) * (y_true - y_pred_mean)**2 + y_pred_log_var
        return torch.mean(loss)

# Data Preparation

In [ ]:
def create_predictive_sequences(features, labels, seq_length):
    """Creates windows through the datasets for model prediction (e.g., 50 seconds to predict 51)"""
    xs, ys, lbls = [], [], []
    if len(features) > seq_length:
        for i in range(len(features) - seq_length):
            xs.append(features[i:(i + seq_length - 1)])
            ys.append(features[i + seq_length - 1])
            lbls.append(labels[i + seq_length - 1])
    return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32), np.array(lbls, dtype=np.float32)

def load_and_prepare_data_by_index(file_path, feature_cols, label_col, seq_length, train_start_index, train_end_index):
    """Creates DataFrame adding new features and separating it"""
    try:
        df = pd.read_csv(file_path, on_bad_lines='skip')
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found")
        return None, None, None, None

    df.columns = df.columns.str.strip()

    if label_col not in df.columns:
        print(f"Warning: Column '{label_col}' not found. Creating a zeroed column (This process will skip the evaluation).")
        df[label_col] = 0

    original_feature_cols = list(feature_cols)
    for col in original_feature_cols:
        if col not in df.columns:
              print(f"Warning:  Column '{col}' not found, skipping...")
              feature_cols.remove(col)

    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df.fillna(df.mean(numeric_only=True), inplace=True)

    if 'total.pckts' in df.columns:
        df['pckts_roc'] = df['total.pckts'].diff().fillna(0)
        if 'pckts_roc' not in feature_cols:
            feature_cols.append('pckts_roc')

    train_df = df.iloc[train_start_index:train_end_index]
    test_df_before = df.iloc[:train_start_index]
    test_df_after = df.iloc[train_end_index:]
    test_df = pd.concat([test_df_before, test_df_after])

    if train_df.empty:
        print("Error: Training df is empty.")
        return None, None, None, None

    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_df[feature_cols])
    test_features_scaled = scaler.transform(test_df[feature_cols]) if not test_df.empty else np.array([])

    X_train, y_train, _ = create_predictive_sequences(train_features_scaled, train_df[label_col].values, seq_length)
    X_test, y_test_targets, y_test_labels = create_predictive_sequences(test_features_scaled, test_df[label_col].values, seq_length)

    return (X_train, y_train, X_test, y_test_targets, y_test_labels), scaler, test_df, feature_cols

# Training and Evaluation

In [ ]:
def train_model(model, train_loader, epochs, learning_rate, weight_decay, max_grad_norm, verbose=True):
    """Trains the Mamba Model"""
    loss_fn = UncertaintyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    model.to(DEVICE)

    if verbose: print("\n Starting Final Model Training...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            mean_pred, log_var_pred = model(X_batch)
            loss = loss_fn(mean_pred, log_var_pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            optimizer.step()
            total_loss += loss.item()

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            avg_loss = total_loss / len(train_loader)
            print(f"Época {epoch+1}/{epochs} - Loss de Previsão: {avg_loss:.6f}")
    if verbose: print("Training finished")
    return model

def evaluate_model(model, data_tuple, scaler, original_test_df, best_params, anomaly_start_index, current_feature_cols, z_score_threshold, feature_to_plot='total.pckts', show_plots=True):
    """Evaluates the model's statistics and plots performance on the dataset"""
    X_train, y_train, X_test, y_test_targets, y_test_labels = data_tuple
    if X_test.shape[0] == 0: return {}

    model.eval()
    model.to(DEVICE)

    with torch.no_grad():
        X_train_t = torch.from_numpy(X_train).to(DEVICE)
        y_train_t = torch.from_numpy(y_train).to(DEVICE)
        train_predictions_mean, train_predictions_log_var = model(X_train_t)
        train_reconstruction_error = torch.exp(-train_predictions_log_var) * (y_train_t - train_predictions_mean)**2
        train_errors = torch.mean(train_reconstruction_error, dim=1).cpu().numpy()

        X_test_t = torch.from_numpy(X_test).to(DEVICE)
        y_test_targets_t = torch.from_numpy(y_test_targets).to(DEVICE)
        test_predictions_mean_scaled, test_predictions_log_var_scaled = model(X_test_t)
        test_reconstruction_error = torch.exp(-test_predictions_log_var_scaled) * (y_test_targets_t - test_predictions_mean_scaled)**2
        raw_anomaly_scores = torch.mean(test_reconstruction_error, dim=1).cpu().numpy()

    anomaly_scores = pd.Series(raw_anomaly_scores).rolling(window=best_params.get('error_window_size', 1), min_periods=1).mean().to_numpy()

    train_error_mean = np.mean(train_errors)
    train_error_std = np.std(train_errors)
    if train_error_std < 1e-6: train_error_std = 1e-6

    test_z_scores = (anomaly_scores - train_error_mean) / train_error_std
    y_pred = (test_z_scores > z_score_threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test_labels, y_pred, labels=[0, 1]).ravel()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    
    precision_w = precision_score(y_test_labels, y_pred, average='weighted', zero_division=0)
    recall_w = recall_score(y_test_labels, y_pred, average='weighted', zero_division=0)

    def format_time(seconds_val):
        if seconds_val is None: return "N/A"
        prefix = "Antecipation" if seconds_val >= 0 else "Delay"
        minutes = abs(seconds_val) // 60
        seconds = abs(seconds_val) % 60
        return f"{prefix}: {int(minutes)}m {int(seconds)}s"

    prediction_time = None
    if original_test_df is not None and not original_test_df.empty:
        first_alert_indices = np.where(y_pred == 1)[0]
        if len(first_alert_indices) > 0 and first_alert_indices[0] + best_params['sequence_length'] - 1 < len(original_test_df.index):
            first_alert_df_index = original_test_df.index[first_alert_indices[0] + best_params['sequence_length'] - 1]
            first_alert_time = anomaly_start_index - first_alert_df_index
            prediction_time = format_time(first_alert_time)

    results = {
        "Dataset": os.path.basename(scaler.source_file_path_) if hasattr(scaler, 'source_file_path_') else 'N/A',
        "Z-Score Thresh": z_score_threshold,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "Accuracy": accuracy,
        "Weighted Precision": precision_w,
        "Weighted Recall": recall_w,
        "Prediction Time": prediction_time,
    }

    plot_feature_name = feature_to_plot
    if plot_feature_name not in current_feature_cols:
        if current_feature_cols:
            plot_feature_name = current_feature_cols[0]
            print(f"\nWarning: Feature ('{feature_to_plot}') not found. Plotting '{plot_feature_name}'.")
        else:
            print("\nWarning: Feature to plot not found. Skipping...")
            plot_feature_name = None

    if show_plots and plot_feature_name and not original_test_df.empty:
        print(f"\n===== EVALUATION WITH Z-SCORE THRESHOLD = {z_score_threshold:.2f} =====")
        print(f"TP: {results['TP']}, TN: {results['TN']}, FP: {results['FP']}, FN: {results['FN']}")
        print(f"Accuracy: {results['Accuracy']:.4f}, Weighted Precision: {results['Weighted Precision']:.4f}, Weighted Recall: {results['Weighted Recall']:.4f}")
        
        with torch.no_grad():
            test_predictions_mean_scaled, _ = model(X_test_t)
        test_predictions_unscaled = scaler.inverse_transform(test_predictions_mean_scaled.cpu().numpy())
        y_test_targets_unscaled = scaler.inverse_transform(y_test_targets)
        feature_index = current_feature_cols.index(plot_feature_name)

        plt.figure(figsize=(15, 6))
        seq_len = best_params['sequence_length']
        test_indices = original_test_df.index[seq_len-1 : len(y_test_targets_unscaled) + seq_len-1]

        plt.scatter(test_indices, y_test_targets_unscaled[:, feature_index], label='Real Traffic', color='blue', alpha=0.5, s=2, zorder=2)
        plt.plot(test_indices, test_predictions_unscaled[:, feature_index], label='Traffic Predicted', color='green', linestyle='-', linewidth=2, zorder=3)

        y_min, y_max = plt.ylim()
        plt.fill_between(test_indices, y_min, y_max, where=(y_test_labels==1), facecolor='orange', alpha=0.3, label='True Anomaly', zorder=1)
        plt.ylim(y_min, y_max)

        anomaly_indices = np.where(y_pred == 1)[0]
        valid_anomaly_indices = [idx for idx in anomaly_indices if idx + seq_len - 1 < len(original_test_df.index)]
        if valid_anomaly_indices:
            anomaly_df_indices = original_test_df.index[np.array(valid_anomaly_indices) + seq_len - 1]
            plt.scatter(anomaly_df_indices, y_test_targets_unscaled[valid_anomaly_indices, feature_index], color='red', label='Anomaly Detected', s=10, zorder=5)

        plt.title(f'Real vs. Predicted for "{plot_feature_name}"')
        plt.xlabel('Time Step')
        plt.ylabel('Feature Value')
        plt.legend()
        plt.grid(True)
        plt.show()

        cm = confusion_matrix(y_test_labels, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Normal', 'Predicted Anomaly'], yticklabels=['Real Normal', 'Real Anomaly'])
        plt.title(f'Confusion Matrix')
        plt.ylabel('Real Class')
        plt.xlabel('Predicted Class')
        plt.show()

    return results

# Hyperparameter Optimization with Optuna

In [ ]:

def objective(trial, exp_config, initial_feature_cols):
    """Optimizes the hyperparameters of the model in an unsupervised method (Validation set)"""
    csv_file = exp_config["csv_file"]
    train_start_index = exp_config.get("train_start_index", 0)
    train_end_index = exp_config["train_end_index"]

    params = {
        'sequence_length': trial.suggest_categorical('sequence_length', [50, 90, 120]),
        'd_model': trial.suggest_categorical('d_model', [32, 64]),
        'd_state': trial.suggest_categorical('d_state', [16, 32]),
        'num_layers': trial.suggest_int('num_layers', 1, 3),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True),
        'epochs': trial.suggest_categorical('epochs', [50, 75]),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64]),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
        'max_grad_norm': trial.suggest_categorical('max_grad_norm', [1.0, 5.0, 10.0]),
    }

    # Training data
    data, _, _, _ = load_and_prepare_data_by_index(
        file_path=csv_file, feature_cols=list(initial_feature_cols), label_col='label',
        seq_length=params['sequence_length'], train_start_index=train_start_index, train_end_index=train_end_index
    )
    if not data or data[0].shape[0] < 2: return float('inf') # Infinity if no data

    X_train_full, y_train_full, _, _, _ = data

    # Train and validation
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=SEED, shuffle=False)

    if len(X_train) == 0 or len(X_val) == 0:
        return float('inf')

    input_dim = X_train.shape[2]
    model = MambaAnomalyDetector(
        input_dim=input_dim, d_model=params['d_model'], d_state=params['d_state'],
        num_layers=params['num_layers'], dropout_rate=params['dropout_rate']
    )
    train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=False)

    trained_model = train_model(
        model, train_loader, params['epochs'], params['learning_rate'],
        params['weight_decay'], params['max_grad_norm'], verbose=False
    )

    # Measures loss on validation 
    loss_fn = UncertaintyLoss()
    trained_model.eval()
    with torch.no_grad():
        X_val_t = torch.from_numpy(X_val).to(DEVICE)
        y_val_t = torch.from_numpy(y_val).to(DEVICE)
        mean_pred, log_var_pred = trained_model(X_val_t)
        validation_loss = loss_fn(mean_pred, log_var_pred, y_val_t).item()

    return validation_loss

# Execution

In [ ]:
if __name__ == '__main__':
    for exp_config in EXPERIMENTS_CONFIG:
        exp_name = exp_config["name"]
        print(f"\n\n{'='*20} INITIALIZING EXPERIMENT: {exp_name.upper()} {'='*20}")

        current_csv_file = exp_config["csv_file"]
        current_model_path = f"mamba_detector_{exp_name}.pth"
        current_params_path = f"best_params_{exp_name}.json"

        try:
            temp_df = pd.read_csv(current_csv_file, on_bad_lines='skip', nrows=1, low_memory=False)
            temp_df.columns = temp_df.columns.str.strip()
            
            TARGET_COLUMN = None
            temp_df_lower_cols = [c.lower() for c in temp_df.columns]

            if 'label' in temp_df_lower_cols:
                TARGET_COLUMN = temp_df.columns[temp_df_lower_cols.index('label')]
            elif 'has_bot' in temp_df_lower_cols:
                TARGET_COLUMN = temp_df.columns[temp_df_lower_cols.index('has_bot')]
            else:
                TARGET_COLUMN = 'label'

            print(f"Anomaly Column: '{TARGET_COLUMN}'")
            
            INITIAL_FEATURE_COLS = [col for col in temp_df.columns if col.lower() not in ['label', 'has_bot', 'unnamed: 0']]

        except FileNotFoundError:
            print(f"Error: File '{current_csv_file}' not found. Skipping")
            continue
        
        data_args = {
            "file_path": current_csv_file, 
            "feature_cols": list(INITIAL_FEATURE_COLS), 
            "label_col": TARGET_COLUMN,
            "train_start_index": exp_config.get("train_start_index", 0),
            "train_end_index": exp_config["train_end_index"],
        }

        if os.path.exists(current_model_path) and os.path.exists(current_params_path):
            print(f"--- Loading model ---")
            with open(current_params_path, 'r') as f: best_params = json.load(f)
            
            data, scaler, original_test_df, final_feature_cols = load_and_prepare_data_by_index(
                seq_length=best_params['sequence_length'], **data_args
            )
            
            if data is None or not data[0].size:
                print(f"Error loading data for'{exp_name}'. Skipping...")
                continue

            trained_final_model = MambaAnomalyDetector(
                input_dim=data[0].shape[2], d_model=best_params['d_model'], d_state=best_params['d_state'],
                num_layers=best_params['num_layers'], dropout_rate=best_params.get('dropout_rate', 0.1)
            )
            trained_final_model.load_state_dict(torch.load(current_model_path))
        else:
            print(f"--- Model not found. Initializing optimization ---")
            study = optuna.create_study(direction='minimize')
            study.optimize(lambda trial: objective(trial, exp_config, INITIAL_FEATURE_COLS), n_trials=exp_config["optuna_trials"])
            
            best_params = study.best_params
            print(f"\nOptimization Finished, best value: {study.best_value:.4f}")
            print(f"Best Hyperparameters: {best_params}")
            with open(current_params_path, 'w') as f: json.dump(best_params, f)

            print("\n--- Training final model with best hyperparameters ---")
            data, scaler, original_test_df, final_feature_cols = load_and_prepare_data_by_index(
                seq_length=best_params['sequence_length'], **data_args
            )
            
            if data is None or not data[0].size:
                print(f"Error loading data for '{exp_name}'. Skipping...")
                continue

            final_model = MambaAnomalyDetector(
                input_dim=data[0].shape[2], d_model=best_params['d_model'], d_state=best_params['d_state'],
                num_layers=best_params['num_layers'], dropout_rate=best_params['dropout_rate']
            )
            train_loader = DataLoader(TensorDataset(torch.from_numpy(data[0]), torch.from_numpy(data[1])), batch_size=best_params['batch_size'], shuffle=True)
            
            trained_final_model = train_model(
                model=final_model, train_loader=train_loader, epochs=best_params['epochs'],
                learning_rate=best_params['learning_rate'], weight_decay=best_params['weight_decay'],
                max_grad_norm=best_params['max_grad_norm'], verbose=True
            )
            torch.save(trained_final_model.state_dict(), current_model_path)
            print(f"Trained model saved on '{current_model_path}'")

        if data:
            experiment_results = []
            print("\n--- Evaluating Final Model ---")
            scaler.source_file_path_ = current_csv_file
            
            adjusted_anomaly_start = exp_config["anomaly_start_index"]
            if exp_config.get("resample_to_minutes", False):
                adjusted_anomaly_start //= 60

            for z in sorted(list(set(exp_config.get("z_scores_to_test"))), reverse=True):
                results = evaluate_model(
                    model=trained_final_model, data_tuple=data, scaler=scaler,
                    original_test_df=original_test_df, best_params=best_params,
                    anomaly_start_index=adjusted_anomaly_start,
                    current_feature_cols=final_feature_cols, z_score_threshold=z,
                    feature_to_plot=exp_config.get("plot_feature", "total.pckts"),
                    show_plots=True
                )
                if results: experiment_results.append(results)

            if experiment_results:
                print(f"\n\n" + "="*25 + f" Experiment Summary: {exp_name.upper()} " + "="*25)
                results_df = pd.DataFrame(experiment_results)
                column_order = ['Dataset', 'Z-Score Thresh', 'TP', 'TN', 'FP', 'FN', 'Accuracy', 'Weighted Precision', 'Weighted Recall', 'Prediction Time']
                final_columns = [col for col in column_order if col in results_df.columns]
                print(results_df[final_columns].to_string())

            print(f"\n{'='*20} EXPERIMENT {exp_name.upper()} FINISHED {'='*20}")
        else:
            print(f"Experiment '{exp_name}' failed.")